# Module 06: Observability (and course wrap-up)

This notebook is the **lab**. The lesson is [`outline.md`](outline.md).

Agents are non-deterministic programs. **Params / metrics / traces** are how you compare two of them. MLflow is the local implementation; Phoenix, Langfuse, and LangSmith are the same buckets with different UIs.

**You do not need `mlflow ui` to finish this module.** We log to a local SQLite file, then query it from Python. The UI is optional and **does not work in Colab**.

Errors: [`instructions.md`](instructions.md).


## Setup

**Local (canonical):** this cell imports `course_setup.py` from the repo root. Your `.env` must live next to `pyproject.toml`.

**Colab:** run the two *Colab only* cells first (install + secrets), then this one. The fallback path uses `HF_TOKEN` from the environment.


In [ ]:
# Colab only — skip this cell locally.
# !pip install -q "smolagents[toolkit,litellm]" python-dotenv pandas requests markdownify huggingface-hub mlflow


In [ ]:
# Colab only — skip this cell locally.
# In Colab: Secrets (key icon) → add HF_TOKEN with "Make calls to Inference Providers".
# import os
# from google.colab import userdata
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")


In [ ]:
import os
import sys
from pathlib import Path

def _repo_root() -> Path | None:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "course_setup.py").exists():
            return candidate
    return None

ROOT = _repo_root()
if ROOT is not None:
    sys.path.insert(0, str(ROOT))
    from course_setup import make_model, print_setup, smoke_test

    model = make_model()
    print_setup(model)
    smoke_test(model)
else:
    from dotenv import load_dotenv
    from smolagents import InferenceClientModel

    load_dotenv()
    token = os.environ.get("HF_TOKEN")
    if not token:
        raise RuntimeError(
            "course_setup.py was not found and HF_TOKEN is unset. "
            "Local: start Jupyter from the cloned repo (`uv run jupyter lab`). "
            "Colab: run the secrets cell, then re-run this cell."
        )
    model = InferenceClientModel(
        model_id=os.environ.get("COURSE_MODEL_ID", "Qwen/Qwen3-Next-80B-A3B-Thinking"),
        token=token,
    )
    print("Model initialized:", model.model_id)


## Local SQLite tracking (no extra terminal)

MLflow 3.16 retired the default filesystem store. SQLite is still local and UI-optional.


In [ ]:
from pathlib import Path
import json
import time
import mlflow
from smolagents import CodeAgent, tool

db = (ROOT / "mlflow.db").resolve()
mlflow.set_tracking_uri(f"sqlite:///{db}")
mlflow.set_experiment("smolagents-course")
print("tracking URI:", mlflow.get_tracking_uri())
print("mlflow", mlflow.__version__)


## Manual log — you should see the three buckets

A tiny `@tool` plus CodeAgent so this module does not depend on the web.


In [ ]:
@tool
def word_count(text: str) -> int:
    """Count whitespace-separated words.

    Args:
        text: Sentence to count.
    """
    return len(text.split())

agent = CodeAgent(tools=[word_count], model=model, max_steps=8)
task = "How many words in 'observability is how we improve agents'? Double it."

with mlflow.start_run(run_name="manual-wordcount"):
    mlflow.log_params({
        "agent_type": "CodeAgent",
        "max_steps": 8,
        "model_id": getattr(model, "model_id", type(model).__name__),
        "task": task[:120],
    })
    t0 = time.perf_counter()
    result = agent.run(task)
    mlflow.log_metrics({
        "duration_seconds": time.perf_counter() - t0,
        "steps_taken": float(len(agent.memory.steps)),
    })
    mlflow.log_text(str(result), "result.txt")
    trace = [
        {"i": i, "type": type(s).__name__, "text": str(s)[:500]}
        for i, s in enumerate(agent.memory.steps)
    ]
    mlflow.log_dict({"steps": trace}, "steps_trace.json")
    print("result:", result)
    print("run_id:", mlflow.active_run().info.run_id if mlflow.active_run() else "(closed)")


## Helper: `run_and_trace`

Same record, reusable. Use this in the exercises.


In [ ]:
def run_and_trace(agent, task: str, run_name: str, extra_params: dict | None = None):
    extra_params = extra_params or {}
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            "agent_type": type(agent).__name__,
            "max_steps": getattr(agent, "max_steps", None),
            "model_id": getattr(getattr(agent, "model", None), "model_id", ""),
            "task": task[:120],
            **extra_params,
        })
        t0 = time.perf_counter()
        result = agent.run(task)
        duration = time.perf_counter() - t0
        mlflow.log_metrics({
            "duration_seconds": duration,
            "steps_taken": float(len(agent.memory.steps)),
        })
        mlflow.log_text(str(result), "result.txt")
        mlflow.log_dict(
            {
                "steps": [
                    {"i": i, "type": type(s).__name__, "text": str(s)[:500]}
                    for i, s in enumerate(agent.memory.steps)
                ]
            },
            "steps_trace.json",
        )
        return result, duration, len(agent.memory.steps)

fresh = CodeAgent(tools=[word_count], model=model, max_steps=8)
res, dur, n = run_and_trace(fresh, task, "helper-wordcount")
print(res, "steps", n, "s", round(dur, 2))


## Autolog (after you understand the record)

One line, then run as usual. Inspect with `search_runs` — no UI required.


In [ ]:
mlflow.smolagents.autolog()
auto_agent = CodeAgent(tools=[word_count], model=model, max_steps=8)
auto_result = auto_agent.run("Count the words in 'autolog should still show a trace'.")
print("autolog result:", auto_result)

runs = mlflow.search_runs(experiment_names=["smolagents-course"])
print(runs[["run_id", "status", "metrics.steps_taken", "params.agent_type"]].head())


## Optional UI

Local only:

```bash
uv run mlflow ui --port 5000 --backend-store-uri sqlite:///mlflow.db
```

Open http://localhost:5000 → experiment `smolagents-course` → compare two runs. In Colab, skip this; `search_runs` above is the equivalent.


## Exercises


In [ ]:
# TODO Exercise 1: trace a manager-style run
# Rebuild a tiny Module 05 manager (data_analyst on data/sample_sales.csv is enough).
# Wrap manager.run(...) with run_and_trace.
# If you can, also mlflow.log_metric("specialist_steps", len(data_analyst.memory.steps)).
#
# You succeeded if:
#   - search_runs shows a run named like "manager-sales"
#   - that run has steps_taken >= 2
#   - an artifact result.txt exists conceptually (you logged it)

# Your code here:


In [ ]:
# TODO Exercise 2: non-determinism, three times
# Same CodeAgent class + same task, three run_and_trace calls with names run-a/b/c.
# Print steps_taken and the three results.
#
# You succeeded if:
#   - three rows appear in search_runs for those names
#   - you wrote one sentence: identical / same facts different wording / disagreed
#   - you did not treat variance as a failed install

# Your code here:


## Hints

<details>
<summary>search_runs filter</summary>

```python
mlflow.search_runs(experiment_names=["smolagents-course"], filter_string="attributes.run_name LIKE 'run-%'")
```

</details>


## What you built (this module)

- A file-store experiment you can query without a server
- Manual logs so you know what a “run” contains
- Autolog as the modern shortcut, with eyes open about its gaps

## What you built (the course)

The loop, the schema, two action languages, retrieval, routing, traces.

**Next outside this repo:** [HF Agents Course](https://huggingface.co/learn/agents-course) · MCP via `ToolCollection.from_mcp` · a 3-task eval harness · a real CodeAgent sandbox.

You are done when you can watch a trace and say *why* the next tool was called — in smolagents or anywhere else.
